# R12 batch - pairwise identity scorers (H121, H122, H123, H125, H126, H143)

Benchmark document set: the interim identity pair set (252 pairs frozen in `reports/matching-r12-foundation-*.json`).
Positives (y=1): variance (47) + resolver_miss (19) + samename (20) = 86. Hard negatives (y=0): sibling (40) + false_merge (6) = 46. Random (120) excluded from the scored metrics.

Two metrics per scorer:
- **Overall AUC** - duplicate (y=1) vs hard negatives (sibling + false_merge)
- **Variance-vs-sibling AUC** - the surface-variance class (47) vs the model-number-family siblings (40)

Baseline: raw Titan cosine, recomputed on this exact frozen pair set and name->id mapping so every scorer is apples-to-apples (registered baseline 0.888; recomputed here for internal consistency).

Hypotheses:
- H121 cross-encoders (bge-reranker-base, ms-marco-MiniLM-L6-v2) - bar: overall AUC gap >= cosine + 0.10
- H122 NLI bidirectional (mDeBERTa-v3-base-mnli-xnli) - bar: asymmetry recovers >= 70% of variance; contradiction fires on >= 60% of siblings
- H123 bi-encoder bake-off (bge-base-en-v1.5 CLS, e5-base-v2 prefix+mean) vs Titan - bar: >= 0.05 over Titan
- H125 instruction embedder (e5-mistral-7b-instruct) - bar: closes >= 50% of the bi-encoder->cross-encoder gap; refuted if it moves AUC < 0.02
- H126 field decomposition (name / spec / description channels, 3-feature logistic 5-fold CV) vs single full-record vector - bar: +0.07
- H143 ensemble (cross-encoder + both NLI directions + contradiction) vs best individual - bar: +0.04; refuted if signals correlate > 0.9

In [1]:
# GPU selection - MUST precede torch import
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "2"  # RTX 5000 Ada 32GB, sm_89
os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [2]:
# Imports
# stdlib
import datetime  # report stamps
import glob  # latest pair set
import json  # report io
import re  # field handling

# third party
import numpy as np  # vectors
import torch  # gpu scorers
from rich import print as rprint
from sklearn.linear_model import LogisticRegression  # H126
from sklearn.model_selection import cross_val_predict, StratifiedKFold  # honest AUC
from sklearn.metrics import roc_auc_score
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from neo4j import GraphDatabase

print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")


/home/lab/workspace/learning/projects/knowledge-graph-foundry/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch 2.12.1+cu130 | cuda True | NVIDIA RTX 5000 Ada Generation


In [3]:
# Reproducibility
import random
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)


In [4]:
# Configuration
NEO4J_URI = "bolt://user-konrad.jelen-kgf-neo4j2:7687"  # READ-ONLY neo4j2
NEO4J_AUTH = ("neo4j", "kgfoundry")
RECORD_CAP = 300  # chars per full record text

OVERALL_POS = {"variance", "resolver_miss", "samename"}
OVERALL_NEG = {"sibling", "false_merge"}

BAR_H121 = 0.10   # cross-encoder overall AUC gap over cosine
BAR_H123 = 0.05   # bi-encoder over Titan
BAR_H125_MIN = 0.02  # instruction must move AUC at least this
BAR_H125_CLOSE = 0.50  # fraction of bi->cross gap closed
BAR_H126 = 0.07   # field-decomposed logistic over single vector
BAR_H143 = 0.04   # ensemble over best individual
BAR_H143_CORR = 0.90  # refuted if signals correlate above this
BAR_H122_VAR = 0.70   # variance recovered by entailment
BAR_H122_SIB = 0.60   # siblings flagged by contradiction

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

rprint(f"""[bold cyan]Configuration[/bold cyan]
[dim]{"-"*46}[/dim]
  Graph (read-only): [cyan]{NEO4J_URI}[/cyan]
  Device: [green]{DEVICE}[/green]
  Bars: H121 gap>={BAR_H121}  H123>={BAR_H123}  H125 close>={BAR_H125_CLOSE}  H126>={BAR_H126}  H143>={BAR_H143}
""")


Configuration
----------------------------------------------
  Graph (read-only): bolt://user-konrad.jelen-kgf-neo4j2:7687
  Device: cuda
  Bars: H121 gap>=0.1  H123>=0.05  H125 close>=0.5  H126>=0.07  H143>=0.04

## Data loading - frozen pair set + entity records

In [5]:
# Load the frozen 252-pair set (names + labels), fetch entity records from neo4j2
rep_path = sorted(glob.glob("../reports/matching-r12-foundation-*.json"))[-1]
rep = json.load(open(rep_path))
PAIRS_RAW = rep["pairs"]  # list of {a,b,y,cls}
print("pair set:", rep_path, "|", len(PAIRS_RAW), "pairs")

driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH, notifications_min_severity="OFF")
with driver.session() as s:
    ents = s.run(
        "MATCH (e:Entity) WHERE e.embedding IS NOT NULL "
        "RETURN e.id AS id, e.name AS name, labels(e) AS types, "
        "e.description AS description, e.embedding AS emb, properties(e) AS props"
    ).data()
driver.close()

name_to_id = {}
for r in ents:
    name_to_id.setdefault(r["name"], r["id"])
REC = {r["id"]: r for r in ents}
TITAN = {r["id"]: np.asarray(r["emb"], dtype=np.float32) for r in ents}
print("entities:", len(REC), "| titan dim:", len(next(iter(TITAN.values()))))


pair set: ../reports/matching-r12-foundation-20260706-211008.json | 252 pairs


entities: 2798 | titan dim: 1024


In [6]:
# Build record text + field channels; assemble aligned pair list P
def fields(rec):
    typ = next((t for t in rec["types"] if t != "Entity"), "Entity")
    name = rec["name"] or ""
    desc = rec["description"] or ""
    specs = []
    for k, v in rec["props"].items():
        if k.startswith("prop_") and v not in (None, "", []):
            specs.append(f"{k[5:]}: {v}")
    spec = "; ".join(specs)
    full = f"{typ}: {name}"
    if desc:
        full += f" - {desc}"
    if spec:
        full += f" | {spec}"
    return dict(type=typ, name=name, desc=desc, spec=spec, full=full[:RECORD_CAP])

FLD = {i: fields(r) for i, r in REC.items()}

P = []  # aligned scored pairs
for p in PAIRS_RAW:
    a, b = name_to_id.get(p["a"]), name_to_id.get(p["b"])
    if a is None or b is None:
        continue
    P.append(dict(a=a, b=b, y=p["y"], cls=p["cls"]))
print("aligned pairs:", len(P))

# unique entities used in pairs (encode once)
UIDS = sorted({x for p in P for x in (p["a"], p["b"])})
print("unique entities in pairs:", len(UIDS))

from collections import Counter
print("class counts:", dict(Counter(p["cls"] for p in P)))


aligned pairs: 252
unique entities in pairs: 413
class counts: {'variance': 47, 'resolver_miss': 19, 'samename': 20, 'false_merge': 6, 'sibling': 40, 'random': 120}


In [7]:
# Metric helpers - AUC over a class subset, aligned to P order
def subset_auc(scores, pos_cls, neg_cls):
    y, s = [], []
    for i, p in enumerate(P):
        if p["cls"] in pos_cls:
            y.append(1); s.append(scores[i])
        elif p["cls"] in neg_cls:
            y.append(0); s.append(scores[i])
    return float(roc_auc_score(y, s))

def overall_auc(scores):
    return subset_auc(scores, OVERALL_POS, OVERALL_NEG)

def varsib_auc(scores):
    return subset_auc(scores, {"variance"}, {"sibling"})

RESULTS = {}  # name -> dict(overall, varsib, scores)
def record(name, scores):
    scores = list(map(float, scores))
    RESULTS[name] = dict(overall=overall_auc(scores), varsib=varsib_auc(scores), scores=scores)
    rprint(f"  [cyan]{name:28s}[/cyan] overall [yellow]{RESULTS[name]['overall']:.4f}[/yellow]  var-vs-sib [yellow]{RESULTS[name]['varsib']:.4f}[/yellow]")
    return RESULTS[name]


## Titan baseline (recomputed on the frozen pairs)

In [8]:
def cos_titan(a, b):
    va, vb = TITAN[a], TITAN[b]
    return float(va @ vb / (np.linalg.norm(va) * np.linalg.norm(vb) + 1e-12))

titan_scores = [cos_titan(p["a"], p["b"]) for p in P]
rprint("[bold]Titan cosine baseline[/bold]")
record("titan_cosine", titan_scores)


Titan cosine baseline

titan_cosine                 overall 0.8931  var-vs-sib 0.9713

{'overall': 0.8930738119312437,
 'varsib': 0.9712765957446808,
 'scores': [0.9877188205709116,
  0.9334227442732059,
  0.9206933379164071,
  0.9129685163488794,
  0.9039952754965325,
  0.9245442748060517,
  0.9190658330908167,
  0.9064323250303935,
  0.9634947158979507,
  0.9084518551817392,
  0.9248042106619169,
  0.9329688549032418,
  0.9082021713247753,
  0.9383792877187881,
  0.9043459835249669,
  0.9583201408376646,
  0.9455518722524723,
  0.9184342026701325,
  0.969768224343774,
  0.9191796183576928,
  0.9409946799268849,
  0.9432359933843716,
  0.9146783351889046,
  0.9552095532407744,
  0.9256293177595418,
  0.9748761653890397,
  0.9655902426721641,
  0.9653528312049291,
  0.9101486206045585,
  0.906367957591104,
  0.9355943760620761,
  0.9220917224874812,
  0.9374577999105614,
  0.9442159533491229,
  0.9170461893072493,
  0.9084510803213571,
  0.9136677928955832,
  0.9283381104460016,
  0.9096379280081235,
  0.9282286763181941,
  0.9169750213613876,
  0.9096741676321469,
  0.9

## H123 - bi-encoder bake-off (bge-base CLS, e5-base prefix+mean) vs Titan

In [9]:
# Bi-encoder helper: encode full-record texts, score pairs by cosine
def biencode_and_score(model_id, prefix="", pooling_note=""):
    st = SentenceTransformer(model_id, device=DEVICE)
    texts = [prefix + FLD[i]["full"] for i in UIDS]
    emb = st.encode(texts, batch_size=32, normalize_embeddings=True,
                    show_progress_bar=False, convert_to_numpy=True)
    idx = {i: emb[k] for k, i in enumerate(UIDS)}
    scores = [float(idx[p["a"]] @ idx[p["b"]]) for p in P]
    del st
    torch.cuda.empty_cache()
    return scores, idx

rprint("[bold]H123 bi-encoders[/bold]")
sc_bge, EMB_BGE = biencode_and_score("BAAI/bge-base-en-v1.5")            # CLS pooling (model config)
record("bge-base-en-v1.5", sc_bge)
sc_e5, EMB_E5 = biencode_and_score("intfloat/e5-base-v2", prefix="query: ")  # mean pooling + query prefix
record("e5-base-v2", sc_e5)


H123 bi-encoders

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:  71%|███████▏  | 142/199 [00:00<00:00, 1414.45it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1766.44it/s]

bge-base-en-v1.5             overall 0.7060  var-vs-sib 0.7489

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:  74%|███████▍  | 147/199 [00:00<00:00, 1459.18it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1767.43it/s]

e5-base-v2                   overall 0.6926  var-vs-sib 0.7713

{'overall': 0.6926188068756319,
 'varsib': 0.7712765957446808,
 'scores': [0.9451682567596436,
  0.9460278749465942,
  0.9481607675552368,
  0.9787790775299072,
  0.9370145797729492,
  0.9356141686439514,
  0.912023663520813,
  0.9462082386016846,
  0.911992073059082,
  0.9161210060119629,
  0.912372350692749,
  0.9206662774085999,
  0.9409670829772949,
  0.9458633661270142,
  0.9575185179710388,
  0.921249508857727,
  0.9746332764625549,
  0.9346447587013245,
  0.982087254524231,
  0.8790464401245117,
  0.9751481413841248,
  0.9513463973999023,
  0.9671617746353149,
  0.9309595227241516,
  0.9600614309310913,
  0.9793263077735901,
  0.9582082033157349,
  0.9089362621307373,
  0.9430718421936035,
  0.9485101699829102,
  0.9285027384757996,
  0.9693384170532227,
  0.9647177457809448,
  0.9477581977844238,
  0.9792439937591553,
  0.9600598812103271,
  0.9898509383201599,
  0.9731850028038025,
  0.9839507341384888,
  0.971956729888916,
  0.9792456030845642,
  0.9794011116027832,
  0.98447

In [10]:
# H123 verdict
titan_o = RESULTS["titan_cosine"]["overall"]
h123_tbl = []
best_bi_name, best_bi = "titan_cosine", titan_o
for m in ["bge-base-en-v1.5", "e5-base-v2"]:
    o = RESULTS[m]["overall"]
    gain = o - titan_o
    h123_tbl.append((m, o, gain))
    if o > best_bi:
        best_bi_name, best_bi = m, o
h123_pass = any(g >= BAR_H123 for _, _, g in h123_tbl)
rprint(f"[bold]H123[/bold] Titan overall={titan_o:.4f}")
for m, o, g in h123_tbl:
    rprint(f"  {m:22s} overall {o:.4f}  gain vs Titan {g:+.4f}  {'PASS' if g>=BAR_H123 else 'below bar'}")
rprint(f"  best bi-encoder: [green]{best_bi_name}[/green] ({best_bi:.4f})  ->  H123 {'CONFIRMED' if h123_pass else 'REFUTED'} (bar +{BAR_H123})")
H123 = dict(titan=titan_o, table=h123_tbl, best_bi=best_bi_name, best_bi_auc=best_bi, verdict="CONFIRMED" if h123_pass else "REFUTED")


H123 Titan overall=0.8931

bge-base-en-v1.5       overall 0.7060  gain vs Titan -0.1871  below bar

e5-base-v2             overall 0.6926  gain vs Titan -0.2005  below bar

best bi-encoder: titan_cosine (0.8931)  ->  H123 REFUTED (bar +0.05)

## H121 - cross-encoders (joint attention over the pair)

In [11]:
# Cross-encoders score (text_a, text_b) jointly; higher = more duplicate-like
def crossencode(model_id):
    ce = CrossEncoder(model_id, device=DEVICE, max_length=512)
    inp = [(FLD[p["a"]]["full"], FLD[p["b"]]["full"]) for p in P]
    sc = ce.predict(inp, batch_size=32, show_progress_bar=False)
    del ce
    torch.cuda.empty_cache()
    return list(map(float, np.asarray(sc).ravel()))

rprint("[bold]H121 cross-encoders[/bold]")
sc_bgece = crossencode("BAAI/bge-reranker-base")
record("bge-reranker-base", sc_bgece)
sc_msm = crossencode("cross-encoder/ms-marco-MiniLM-L6-v2")
record("ms-marco-MiniLM-L6", sc_msm)


H121 cross-encoders

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:  69%|██████▊   | 138/201 [00:00<00:00, 1371.90it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1716.87it/s]

bge-reranker-base            overall 0.8511  var-vs-sib 0.8128

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1587.35it/s]

ms-marco-MiniLM-L6           overall 0.7020  var-vs-sib 0.7750

{'overall': 0.7019716885743175,
 'varsib': 0.775,
 'scores': [5.26814079284668,
  4.462838172912598,
  7.69867467880249,
  8.835309028625488,
  3.5319132804870605,
  6.118002891540527,
  3.7899630069732666,
  6.471524238586426,
  9.277679443359375,
  4.996371269226074,
  4.232104778289795,
  5.584826469421387,
  5.257816791534424,
  5.620058536529541,
  6.590329170227051,
  4.141379356384277,
  9.258394241333008,
  6.656019687652588,
  6.106621742248535,
  1.942976713180542,
  7.001477241516113,
  6.832415580749512,
  7.623322486877441,
  7.897503852844238,
  8.01628589630127,
  7.672877788543701,
  8.005998611450195,
  4.541065216064453,
  3.332252025604248,
  4.473938941955566,
  8.448409080505371,
  8.683506965637207,
  7.841176986694336,
  6.560669422149658,
  5.626158714294434,
  7.735830307006836,
  9.09463882446289,
  8.612911224365234,
  7.364071846008301,
  8.247297286987305,
  8.808337211608887,
  7.532550811767578,
  5.823063850402832,
  6.885838508605957,
  6.61095285415649

In [12]:
# H121 verdict - gap over cosine baseline
cos_o = RESULTS["titan_cosine"]["overall"]
h121_tbl = []
best_ce_name, best_ce = None, -1
for m in ["bge-reranker-base", "ms-marco-MiniLM-L6"]:
    o = RESULTS[m]["overall"]
    h121_tbl.append((m, o, o - cos_o, RESULTS[m]["varsib"]))
    if o > best_ce:
        best_ce_name, best_ce = m, o
h121_pass = any(gap >= BAR_H121 for _, _, gap, _ in h121_tbl)
rprint(f"[bold]H121[/bold] cosine overall={cos_o:.4f}  bar=cosine+{BAR_H121}={cos_o+BAR_H121:.4f}")
for m, o, gap, vs in h121_tbl:
    rprint(f"  {m:22s} overall {o:.4f}  gap {gap:+.4f}  var-vs-sib {vs:.4f}  {'PASS' if gap>=BAR_H121 else 'below bar'}")
rprint(f"  H121 {'CONFIRMED' if h121_pass else 'REFUTED'}")
H121 = dict(cosine=cos_o, bar=cos_o+BAR_H121, table=h121_tbl, best_ce=best_ce_name, best_ce_auc=best_ce, verdict="CONFIRMED" if h121_pass else "REFUTED")


H121 cosine overall=0.8931  bar=cosine+0.1=0.9931

bge-reranker-base      overall 0.8511  gap -0.0420  var-vs-sib 0.8128  below bar

ms-marco-MiniLM-L6     overall 0.7020  gap -0.1911  var-vs-sib 0.7750  below bar

H121 REFUTED

## H122 - NLI bidirectional (entailment asymmetry, contradiction on siblings)

In [13]:
# mDeBERTa 3-label NLI; read id2label for the entailment/contradiction indices
NLI_ID = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
nli_tok = AutoTokenizer.from_pretrained(NLI_ID)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_ID).to(DEVICE).eval()
id2label = nli_model.config.id2label
print("id2label:", id2label)
lab2id = {v.lower(): k for k, v in id2label.items()}
ENT_IX = next(k for k, v in id2label.items() if "entail" in v.lower())
CON_IX = next(k for k, v in id2label.items() if "contradict" in v.lower())
print("entailment index:", ENT_IX, "| contradiction index:", CON_IX)

@torch.no_grad()
def nli_probs(premises, hypotheses, bs=32):
    out = []
    for i in range(0, len(premises), bs):
        enc = nli_tok(premises[i:i+bs], hypotheses[i:i+bs], truncation=True,
                      max_length=256, padding=True, return_tensors="pt").to(DEVICE)
        logits = nli_model(**enc).logits
        out.append(torch.softmax(logits, dim=-1).cpu().numpy())
    return np.concatenate(out, 0)

ta = [FLD[p["a"]]["full"] for p in P]
tb = [FLD[p["b"]]["full"] for p in P]
prob_ab = nli_probs(ta, tb)  # premise a, hypothesis b
prob_ba = nli_probs(tb, ta)  # premise b, hypothesis a
e_ab, e_ba = prob_ab[:, ENT_IX], prob_ba[:, ENT_IX]
c_ab, c_ba = prob_ab[:, CON_IX], prob_ba[:, CON_IX]
print("NLI scored", len(P), "pairs x2 directions")


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Loading weights:  46%|████▌     | 92/202 [00:00<00:00, 898.26it/s]

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 1046.74it/s]

id2label: {0: 'entailment', 1: 'neutral', 2: 'contradiction'}
entailment index: 0 | contradiction index: 2


NLI scored 252 pairs x2 directions


In [14]:
# NLI-derived scorers for the leaderboard
nli_mean_ent = ((e_ab + e_ba) / 2).tolist()
nli_max_ent = np.maximum(e_ab, e_ba).tolist()
nli_min_ent = np.minimum(e_ab, e_ba).tolist()  # mutual entailment
rprint("[bold]H122 NLI scorers[/bold]")
record("nli_mean_entail", nli_mean_ent)
record("nli_max_entail", nli_max_ent)
record("nli_mutual_entail", nli_min_ent)

# H122 mechanism bars
THR = 0.5
var_mask = np.array([p["cls"] == "variance" for p in P])
sib_mask = np.array([p["cls"] == "sibling" for p in P])
maxe = np.maximum(e_ab, e_ba)
maxc = np.maximum(c_ab, c_ba)
asym = np.abs(e_ab - e_ba)

var_recovered = float(((maxe[var_mask] >= THR)).mean())            # entailment present in >=1 direction
var_asym = float(((maxe[var_mask] >= THR) & (np.minimum(e_ab, e_ba)[var_mask] < THR)).mean())  # one-directional
sib_contra = float((maxc[sib_mask] >= THR).mean())                 # contradiction fires
sib_contra_argmax = float(np.array([(np.argmax(prob_ab[i]) == CON_IX) or (np.argmax(prob_ba[i]) == CON_IX) for i in range(len(P)) if P[i]["cls"] == "sibling"]).mean())

rprint(f"  variance recovered by entailment (max>= {THR}): [yellow]{var_recovered:.2%}[/yellow]  (bar {BAR_H122_VAR:.0%})")
rprint(f"  variance one-directional (asymmetric): [yellow]{var_asym:.2%}[/yellow]")
rprint(f"  siblings with contradiction (max>= {THR}): [yellow]{sib_contra:.2%}[/yellow]  (bar {BAR_H122_SIB:.0%})")
rprint(f"  siblings contradiction=argmax: [yellow]{sib_contra_argmax:.2%}[/yellow]")
h122_pass = (var_recovered >= BAR_H122_VAR) and (sib_contra >= BAR_H122_SIB)
rprint(f"  H122 {'CONFIRMED' if h122_pass else 'REFUTED'} (both clauses required)")
H122 = dict(var_recovered=var_recovered, var_asym=var_asym, sib_contra=sib_contra,
            sib_contra_argmax=sib_contra_argmax, verdict="CONFIRMED" if h122_pass else "REFUTED")


H122 NLI scorers

nli_mean_entail              overall 0.6364  var-vs-sib 0.5816

nli_max_entail               overall 0.6265  var-vs-sib 0.5649

nli_mutual_entail            overall 0.6368  var-vs-sib 0.6309

variance recovered by entailment (max>= 0.5): 25.53%  (bar 70%)

variance one-directional (asymmetric): 12.77%

siblings with contradiction (max>= 0.5): 90.00%  (bar 60%)

siblings contradiction=argmax: 92.50%

H122 REFUTED (both clauses required)

## H126 - field-decomposed likelihoods (name / spec / description channels)

In [15]:
# Encode each field channel separately with the H123 winning bi-encoder
win_id = {"bge-base-en-v1.5": ("BAAI/bge-base-en-v1.5", ""),
          "e5-base-v2": ("intfloat/e5-base-v2", "query: "),
          "titan_cosine": ("BAAI/bge-base-en-v1.5", "")}[H123["best_bi"] if H123["best_bi"] in ("bge-base-en-v1.5","e5-base-v2") else "bge-base-en-v1.5"]
FIELD_MODEL, FIELD_PREFIX = win_id
rprint(f"H126 field embedder: [green]{FIELD_MODEL}[/green]")

stf = SentenceTransformer(FIELD_MODEL, device=DEVICE)
def field_channel(fname):
    txts = [FIELD_PREFIX + (FLD[i][fname] or "") for i in UIDS]
    emb = stf.encode(txts, batch_size=32, normalize_embeddings=True, show_progress_bar=False, convert_to_numpy=True)
    idx = {i: emb[k] for k, i in enumerate(UIDS)}
    return np.array([float(idx[p["a"]] @ idx[p["b"]]) for p in P])

name_sim = field_channel("name")
spec_sim = field_channel("spec")
desc_sim = field_channel("desc")
full_sim = field_channel("full")  # single full-record vector (same embedder, fair control)
del stf; torch.cuda.empty_cache()
print("field channels computed")


H126 field embedder: BAAI/bge-base-en-v1.5

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5597.43it/s]

field channels computed


In [16]:
# 3-feature logistic vs single full-record vector, honest 5-fold CV on the scored subset
def scored_subset():
    idxs, y = [], []
    for i, p in enumerate(P):
        if p["cls"] in OVERALL_POS:
            idxs.append(i); y.append(1)
        elif p["cls"] in OVERALL_NEG:
            idxs.append(i); y.append(0)
    return np.array(idxs), np.array(y)

sub, ysub = scored_subset()
X3 = np.column_stack([name_sim[sub], spec_sim[sub], desc_sim[sub]])
X1 = full_sim[sub].reshape(-1, 1)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
p3 = cross_val_predict(LogisticRegression(max_iter=1000), X3, ysub, cv=cv, method="predict_proba")[:, 1]
p1 = cross_val_predict(LogisticRegression(max_iter=1000), X1, ysub, cv=cv, method="predict_proba")[:, 1]
auc3 = float(roc_auc_score(ysub, p3))
auc1 = float(roc_auc_score(ysub, p1))

# variance-vs-sibling diagnostic on the field channels
def vs_of(sim):
    return subset_auc(list(sim), {"variance"}, {"sibling"})
rprint("[bold]H126 field decomposition[/bold]")
rprint(f"  single full-record logistic AUC: [yellow]{auc1:.4f}[/yellow]")
rprint(f"  3-channel (name+spec+desc) logistic AUC: [yellow]{auc3:.4f}[/yellow]  gain [yellow]{auc3-auc1:+.4f}[/yellow]  (bar +{BAR_H126})")
rprint(f"  channel var-vs-sib AUC:  name {vs_of(name_sim):.3f}  spec {vs_of(spec_sim):.3f}  desc {vs_of(desc_sim):.3f}")
# diagnostic pattern: mean channel sims per class
for cls in ["variance", "sibling"]:
    m = np.array([p["cls"] == cls for p in P])
    rprint(f"  [{cls}] mean name {name_sim[m].mean():.3f}  spec {spec_sim[m].mean():.3f}  desc {desc_sim[m].mean():.3f}")
h126_pass = (auc3 - auc1) >= BAR_H126
rprint(f"  H126 {'CONFIRMED' if h126_pass else 'REFUTED'}")
H126 = dict(auc_single=auc1, auc_3field=auc3, gain=auc3-auc1,
            channel_varsib=dict(name=vs_of(name_sim), spec=vs_of(spec_sim), desc=vs_of(desc_sim)),
            verdict="CONFIRMED" if h126_pass else "REFUTED")


H126 field decomposition

single full-record logistic AUC: 0.6954

3-channel (name+spec+desc) logistic AUC: 0.6951  gain -0.0003  (bar +0.07)

channel var-vs-sib AUC:  name 0.638  spec 0.554  desc 0.693

mean name 0.886  spec 0.822  desc 0.893

mean name 0.874  spec 0.806  desc 0.768

H126 REFUTED

## H125 - instruction-tuned identity representation (e5-mistral-7b-instruct)

In [17]:
# Instruction embedder: condition the vector on an identity-matching task string
H125_MODEL = "intfloat/e5-mistral-7b-instruct"
H125_TASK = "Represent this product listing for identity matching, ignoring marketing descriptors"
h125_scores, h125_err = None, None
try:
    st = SentenceTransformer(H125_MODEL, device=DEVICE, model_kwargs={"torch_dtype": torch.float16})
    def instruct(t):
        return f"Instruct: {H125_TASK}\nQuery: {t}"
    texts = [instruct(FLD[i]["full"]) for i in UIDS]
    emb = st.encode(texts, batch_size=8, normalize_embeddings=True, show_progress_bar=False, convert_to_numpy=True)
    idx = {i: emb[k] for k, i in enumerate(UIDS)}
    h125_scores = [float(idx[p["a"]] @ idx[p["b"]]) for p in P]
    used = H125_MODEL
    del st; torch.cuda.empty_cache()
except Exception as e:
    h125_err = f"{type(e).__name__}: {e}"
    print("e5-mistral failed:", h125_err, "-> falling back to hkunlp/instructor-base")
    torch.cuda.empty_cache()
    try:
        st = SentenceTransformer("hkunlp/instructor-base", device=DEVICE)
        texts = [[H125_TASK, FLD[i]["full"]] for i in UIDS]
        emb = st.encode(texts, batch_size=16, normalize_embeddings=True, show_progress_bar=False, convert_to_numpy=True)
        idx = {i: emb[k] for k, i in enumerate(UIDS)}
        h125_scores = [float(idx[p["a"]] @ idx[p["b"]]) for p in P]
        used = "hkunlp/instructor-base"
        del st; torch.cuda.empty_cache()
    except Exception as e2:
        h125_err = f"both failed; instructor: {type(e2).__name__}: {e2}"
        used = None
print("H125 model used:", used)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:  57%|█████▋    | 165/290 [00:00<00:00, 1647.68it/s]

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1787.86it/s]

H125 model used: intfloat/e5-mistral-7b-instruct


In [18]:
# H125 verdict - gap closure between best bi-encoder and best cross-encoder
if h125_scores is not None:
    record("instruct_" + (used.split("/")[-1]), h125_scores)
    h125_o = overall_auc(h125_scores)
    bi = best_bi          # best bi-encoder overall (incl Titan) from H123
    ce = best_ce          # best cross-encoder overall from H121
    gap = ce - bi
    closed = (h125_o - bi) / gap if abs(gap) > 1e-9 else float("nan")
    moved = h125_o - bi
    rprint("[bold]H125 instruction embedder[/bold]")
    rprint(f"  best bi-encoder {best_bi_name}={bi:.4f}  best cross-encoder {best_ce_name}={ce:.4f}  gap={gap:+.4f}")
    rprint(f"  instruction overall AUC={h125_o:.4f}  moved {moved:+.4f} vs bi  gap closed {closed:.1%}")
    h125_pass = (abs(moved) >= BAR_H125_MIN) and (gap > 0) and (closed >= BAR_H125_CLOSE)
    if gap <= 0:
        note = "gap non-positive (cross-encoder did not beat bi-encoder) - closure undefined; judged on whether instruction moved AUC"
        h125_pass = moved >= BAR_H125_MIN and False  # cannot close a non-existent gap
    else:
        note = ""
    rprint(f"  H125 {'CONFIRMED' if h125_pass else 'REFUTED'} (moved>= {BAR_H125_MIN} and closes>= {BAR_H125_CLOSE:.0%} of a positive gap). {note}")
    H125 = dict(model=used, overall=h125_o, best_bi=bi, best_ce=ce, gap=gap, moved=moved,
                closed=closed if gap > 0 else None, verdict="CONFIRMED" if h125_pass else "REFUTED", note=note)
else:
    rprint(f"[red]H125 could not run: {h125_err}[/red]")
    H125 = dict(model=None, error=h125_err, verdict="INCONCLUSIVE")


instruct_e5-mistral-7b-instruct overall 0.5493  var-vs-sib 0.5926

H125 instruction embedder

best bi-encoder titan_cosine=0.8931  best cross-encoder bge-reranker-base=0.8511  gap=-0.0420

instruction overall AUC=0.5493  moved -0.3438 vs bi  gap closed 819.3%

H125 REFUTED (moved>= 0.02 and closes>= 50% of a positive gap). gap non-positive (cross-encoder did not beat 
bi-encoder) - closure undefined; judged on whether instruction moved AUC

## H143 - ensemble complementarity (cross-encoder + NLI directions + contradiction)

In [19]:
# Ensemble features: best cross-encoder + e_ab + e_ba + contradiction(max). Logistic, 5-fold CV.
sub, ysub = scored_subset()
best_ce_scores = np.array(RESULTS[best_ce_name]["scores"])
feat_names = [best_ce_name, "e_ab", "e_ba", "contra_max"]
FE = np.column_stack([best_ce_scores, np.array(e_ab), np.array(e_ba), np.maximum(c_ab, c_ba)])
Xe = FE[sub]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
pe = cross_val_predict(LogisticRegression(max_iter=1000), Xe, ysub, cv=cv, method="predict_proba")[:, 1]
auc_ens = float(roc_auc_score(ysub, pe))

# individual AUCs on the same scored subset
def sub_auc_vec(vec):
    return float(roc_auc_score(ysub, np.array(vec)[sub]))
ind = {best_ce_name: sub_auc_vec(best_ce_scores),
       "nli_mean_entail": sub_auc_vec(nli_mean_ent),
       "nli_mutual_entail": sub_auc_vec(nli_min_ent),
       "contra_max(neg)": sub_auc_vec(-np.maximum(c_ab, c_ba))}
best_ind_name = max(ind, key=ind.get)
best_ind = ind[best_ind_name]

# signal correlation (Spearman) among the ensemble columns
from scipy.stats import spearmanr
corr = spearmanr(Xe).correlation
rprint("[bold]H143 ensemble[/bold]")
rprint(f"  ensemble 5-fold AUC: [yellow]{auc_ens:.4f}[/yellow]")
for k, v in ind.items():
    rprint(f"  individual {k:22s} {v:.4f}")
rprint(f"  best individual: [green]{best_ind_name}[/green] {best_ind:.4f}  ensemble gain {auc_ens-best_ind:+.4f}  (bar +{BAR_H143})")
maxcorr = float(np.nanmax(np.abs(corr - np.eye(len(feat_names)))))
rprint(f"  max |Spearman| between signals: {maxcorr:.3f}  (refute if > {BAR_H143_CORR})")

# error-set overlap: cross-encoder vs NLI on the scored subset at Youden threshold
def errset(vec):
    v = np.array(vec)[sub]
    fpr_tpr = sorted(set(v))
    # threshold maximizing Youden J
    best_j, best_t = -1, 0
    for t in fpr_tpr:
        pred = (v >= t).astype(int)
        tp = ((pred == 1) & (ysub == 1)).sum(); fn = ((pred == 0) & (ysub == 1)).sum()
        tn = ((pred == 0) & (ysub == 0)).sum(); fp = ((pred == 1) & (ysub == 0)).sum()
        j = tp/(tp+fn+1e-9) + tn/(tn+fp+1e-9) - 1
        if j > best_j:
            best_j, best_t = j, t
    return set(np.where((v >= best_t).astype(int) != ysub)[0])

err_ce = errset(best_ce_scores)
err_nli = errset(nli_mean_ent)
inter = len(err_ce & err_nli)
union = len(err_ce | err_nli)
jacc = inter / union if union else 0.0
rprint(f"  error sets @Youden: cross-encoder {len(err_ce)} errs, NLI {len(err_nli)} errs, shared {inter}, Jaccard {jacc:.2f}")
h143_pass = (auc_ens - best_ind >= BAR_H143) and (maxcorr <= BAR_H143_CORR)
rprint(f"  H143 {'CONFIRMED' if h143_pass else 'REFUTED'}")
H143 = dict(auc_ens=auc_ens, individual=ind, best_ind=best_ind_name, best_ind_auc=best_ind,
            gain=auc_ens-best_ind, max_corr=maxcorr, err_ce=len(err_ce), err_nli=len(err_nli),
            err_shared=inter, err_jaccard=jacc, verdict="CONFIRMED" if h143_pass else "REFUTED")


H143 ensemble

ensemble 5-fold AUC: 0.8547

individual bge-reranker-base      0.8511

individual nli_mean_entail        0.6364

individual nli_mutual_entail      0.6368

individual contra_max(neg)        0.7422

best individual: bge-reranker-base 0.8511  ensemble gain +0.0035  (bar +0.04)

max |Spearman| between signals: 0.765  (refute if > 0.9)

error sets @Youden: cross-encoder 26 errs, NLI 64 errs, shared 18, Jaccard 0.25

H143 REFUTED

## Leaderboard + report

In [20]:
# Leaderboard - every scorer, both metrics
rows = sorted(RESULTS.items(), key=lambda kv: -kv[1]["overall"])
rprint("[bold cyan]Leaderboard (overall = dup vs hard-neg; var-vs-sib = variance vs sibling)[/bold cyan]")
rprint(f"[dim]{'scorer':30s} {'overall':>9s} {'var-vs-sib':>11s}[/dim]")
for name, r in rows:
    rprint(f"  {name:30s} {r['overall']:9.4f} {r['varsib']:11.4f}")

stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d-%H%M%S")
out = dict(
    pair_set=rep_path, n_pairs=len(P),
    metrics={n: dict(overall=r["overall"], varsib=r["varsib"]) for n, r in RESULTS.items()},
    H121=H121, H122=H122, H123=H123, H125=H125, H126=H126, H143=H143,
)
outp = f"../reports/matching-scorers-r12-{stamp}.json"
json.dump(out, open(outp, "w"), indent=1, default=float)
print("wrote", outp)


Leaderboard (overall = dup vs hard-neg; var-vs-sib = variance vs sibling)

scorer                           overall  var-vs-sib

titan_cosine                      0.8931      0.9713

bge-reranker-base                 0.8511      0.8128

bge-base-en-v1.5                  0.7060      0.7489

...

nli_mutual_entail                 0.6368      0.6309

nli_mean_entail                   0.6364      0.5816

nli_max_entail                    0.6265      0.5649

instruct_e5-mistral-7b-instruct    0.5493      0.5926

wrote ../reports/matching-scorers-r12-20260707-091155.json
